# FMA Data Preprocessing

This notebook cleans the FMA metadata and writes **both** small and medium CSV outputs every run. `DATASET_SIZE` only selects which subset gets spectrograms.

Set the three input values in the first code cell before running:
- `DATASET_DIR_INPUT` — where the FMA dataset lives. This can be the project/data root that contains `fma_metadata/`, or the `fma_small/`, `fma_medium/`, or `fma_metadata/` folder itself.
- `DATASET_SIZE` — `"small"` or `"medium"`; this controls the spectrogram subset only.
- `PROCESSED_DIR_INPUT` — where the processed CSVs and optional `spectrograms/` folder should be saved. Relative paths are resolved under the dataset root; leave blank to use `<dataset root>/fma_preprocessed`.

Subset behavior:
- **CSV outputs** — both `tracks_clean_small*.csv` and `tracks_clean_medium*.csv` are always written.
- **small spectrograms** — selected with `DATASET_SIZE = "small"`; uses `fma_small/` when available, falling back to `fma_medium/` if that is the only audio folder present.
- **medium spectrograms** — selected with `DATASET_SIZE = "medium"`; requires `fma_medium/`. If only `fma_small/` is present, the medium CSVs are still written but medium spectrogram extraction is skipped.


## 1. Setup

Run once if any of these are missing:

```python
%pip install torch torchaudio pandas numpy tqdm soundfile
```

Edit `DATASET_DIR_INPUT`, `DATASET_SIZE`, and `PROCESSED_DIR_INPUT` in the next cell before running the notebook. `DATASET_SIZE` chooses the spectrogram subset; both cleaned CSV sets are always written.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torchaudio

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable

# -------------------------
# User inputs
# -------------------------
# Point this at the folder containing fma_metadata/ plus fma_small/ or fma_medium/.
# You can also point directly at fma_small/, fma_medium/, or fma_metadata/.
# Leave blank to auto-discover from the current notebook directory.
DATASET_DIR_INPUT = ""  # e.g. "/home/lily/acml-project"

# Choose which subset should get spectrograms. CSVs are written for both subsets.
DATASET_SIZE = "small"  # "small" or "medium"

# Choose where processed CSVs and spectrograms should be saved.
# Relative paths are resolved under the dataset root. Leave blank to write to
# <dataset root>/fma_preprocessed.
PROCESSED_DIR_INPUT = ""  # e.g. "fma_processed" or "/home/lily/acml-project/fma_processed"

VALID_DATASET_SIZES = {"small", "medium"}
DATASET_SIZE = DATASET_SIZE.strip().lower()
if DATASET_SIZE not in VALID_DATASET_SIZES:
    raise ValueError('DATASET_SIZE must be either "small" or "medium".')


def _unique_paths(paths):
    seen = set()
    unique = []
    for path in paths:
        resolved = Path(path).expanduser().resolve()
        if resolved not in seen:
            seen.add(resolved)
            unique.append(resolved)
    return unique


def _candidate_dataset_roots(dataset_dir_input):
    if dataset_dir_input:
        given = Path(dataset_dir_input).expanduser()
        return _unique_paths([given, given.parent])
    return _unique_paths([Path.cwd(), *Path.cwd().parents])


def resolve_dataset_root(dataset_dir_input):
    checked = []
    for candidate in _candidate_dataset_roots(dataset_dir_input):
        checked.append(candidate)
        if (candidate / "fma_metadata" / "tracks.csv").exists():
            return candidate
    checked_text = "\n".join(f"  - {path}" for path in checked)
    raise FileNotFoundError(
        "Could not find fma_metadata/tracks.csv. Set DATASET_DIR_INPUT to the FMA "
        "dataset root, fma_metadata/, fma_small/, or fma_medium/. Checked:\n"
        f"{checked_text}"
    )


def existing_audio_dir(dataset_root, dataset_size):
    audio_dir = dataset_root / f"fma_{dataset_size}"
    return audio_dir.resolve() if audio_dir.exists() else None


PROJECT_DIR = resolve_dataset_root(DATASET_DIR_INPUT)
METADATA_PATH = (PROJECT_DIR / "fma_metadata" / "tracks.csv").resolve()
SMALL_AUDIO_DIR = existing_audio_dir(PROJECT_DIR, "small")
MEDIUM_AUDIO_DIR = existing_audio_dir(PROJECT_DIR, "medium")

if DATASET_SIZE == "small":
    SPECTROGRAM_AUDIO_DIR = SMALL_AUDIO_DIR or MEDIUM_AUDIO_DIR
else:
    SPECTROGRAM_AUDIO_DIR = MEDIUM_AUDIO_DIR

if PROCESSED_DIR_INPUT:
    processed_dir_input = Path(PROCESSED_DIR_INPUT).expanduser()
    if not processed_dir_input.is_absolute():
        processed_dir_input = PROJECT_DIR / processed_dir_input
    PROCESSED_DIR = processed_dir_input.resolve()
else:
    PROCESSED_DIR = PROJECT_DIR / "fma_preprocessed"

SPECTROGRAM_DIR = PROCESSED_DIR / "spectrograms"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
SPECTROGRAM_DIR.mkdir(parents=True, exist_ok=True)

EXTRACT_SPECTROGRAMS = SPECTROGRAM_AUDIO_DIR is not None
if DATASET_SIZE == "medium" and MEDIUM_AUDIO_DIR is None:
    print("fma_medium/ was not found; medium spectrogram extraction will be skipped.")

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print({
    "DATASET_SIZE": DATASET_SIZE,
    "PROJECT_DIR": str(PROJECT_DIR),
    "SMALL_AUDIO_DIR": None if SMALL_AUDIO_DIR is None else str(SMALL_AUDIO_DIR),
    "MEDIUM_AUDIO_DIR": None if MEDIUM_AUDIO_DIR is None else str(MEDIUM_AUDIO_DIR),
    "SPECTROGRAM_AUDIO_DIR": None if SPECTROGRAM_AUDIO_DIR is None else str(SPECTROGRAM_AUDIO_DIR),
    "METADATA_PATH": str(METADATA_PATH),
    "PROCESSED_DIR": str(PROCESSED_DIR),
    "SPECTROGRAM_DIR": str(SPECTROGRAM_DIR),
    "EXTRACT_SPECTROGRAMS": EXTRACT_SPECTROGRAMS,
    "device": device,
})


## 2. Load `tracks.csv`

`tracks.csv` uses a two-row header, so columns come back as a `MultiIndex` like `("track", "genre_top")`. We pull only the fields we need and flatten them into a normal DataFrame. `audio_path` is built from the available audio folder for each row: small rows prefer `fma_small/`, while medium rows use `fma_medium/` when it exists.


In [ ]:
def audio_dir_for_subset(subset: str) -> Path | None:
    if subset == "small":
        return SMALL_AUDIO_DIR or MEDIUM_AUDIO_DIR
    if subset == "medium":
        return MEDIUM_AUDIO_DIR
    return None


def track_id_to_audio_path(track_id: int, subset: str) -> str:
    audio_dir = audio_dir_for_subset(subset)
    if audio_dir is None:
        return ""
    track_id = int(track_id)
    filename = f"{track_id:06d}.mp3"
    return str(audio_dir / filename[:3] / filename)


tracks = pd.read_csv(METADATA_PATH, index_col=0, header=[0, 1])

# tracks.index is named "track_id" (from index_col=0). Drop it before building metadata_all
# so the resulting frame has track_id only as a column, not also as an index name -
# otherwise sort_values("track_id") later raises "both an index level and a column label".
tracks_flat = tracks.reset_index(drop=True)

metadata_all = pd.DataFrame({
    "track_id": tracks.index.astype(int).to_numpy(),
    "subset": tracks_flat[("set", "subset")].to_numpy(),
    "split": tracks_flat[("set", "split")].to_numpy(),
    "genre": tracks_flat[("track", "genre_top")].to_numpy(),
    "duration": tracks_flat[("track", "duration")].to_numpy(),
    "bit_rate": tracks_flat[("track", "bit_rate")].to_numpy(),
    "title": tracks_flat[("track", "title")].to_numpy(),
    "artist": tracks_flat[("artist", "name")].to_numpy(),
})
metadata_all["audio_path"] = [
    track_id_to_audio_path(track_id, subset)
    for track_id, subset in zip(metadata_all["track_id"], metadata_all["subset"])
]

print(f"Total tracks in tracks.csv: {len(metadata_all):,}")
print(metadata_all["subset"].value_counts())
metadata_all.head()


## 3. Clean both subsets

Both cleaned CSV sets are produced every run. Each subset drops the same core entries:

- **null** — tracks with no `genre_top`.
- **failed** — track ids `creation.ipynb` already flagged as broken (ffmpeg/header issues, librosa "filter pass-band lies beyond Nyquist", etc.).
- **empty** — tracks whose MP3 is missing on disk or zero bytes, when audio is available for that subset.

Small cleaning checks every selected small MP3. Medium cleaning keeps the FMA convention that medium includes both small and medium tracks; if small audio is available, it checks the small portion for missing/empty audio while retaining medium-only rows for tabular models.


In [ ]:
# Same `keep` helper used in creation.ipynb.
def keep(mask, df, label):
    old = len(df)
    df = df[mask]
    print(f"{label:42s} {old - len(df):>5d} dropped, {len(df):>5d} left")
    return df


def is_file_nonempty(path) -> bool:
    if not path:
        return False
    try:
        path = Path(path)
        return path.exists() and path.stat().st_size > 0
    except OSError:
        return False


# Track ids that creation.ipynb documents as feature-extraction failures
# (ffmpeg header issues, librosa "filter pass-band lies beyond Nyquist", etc.).
FAILED = [
    1440, 26436, 28106, 29166, 29167, 29168, 29169, 29170, 29171, 29172,
    29173, 29179, 38903, 43903, 56757, 57603, 59361, 62095, 62954, 62956,
    62957, 62959, 62971, 75461, 80015, 86079, 92345, 92346, 92347, 92348,
    92349, 92350, 92351, 92352, 92353, 92354, 92355, 92356, 92357, 92358,
    92359, 92360, 92361, 96426, 104623, 106719, 109714, 114448, 114501, 114528,
    115235, 117759, 118003, 118004, 127827, 130296, 130298, 131076, 135804, 136486,
    144769, 144770, 144771, 144773, 144774, 144775, 144776, 144777, 144778, 152204,
    154923,
]


print("=== Cleaning small ===")
metadata_small = metadata_all.copy()

metadata_small = keep(metadata_small["subset"] == "small", metadata_small, "subset == small")
metadata_small = keep(metadata_small["genre"].notna(), metadata_small, "genre not null")
metadata_small = keep(metadata_small["audio_path"].map(is_file_nonempty), metadata_small, "audio file present and non-empty")
metadata_small = keep(~metadata_small["track_id"].isin(FAILED), metadata_small, "not in FAILED list")

metadata_small = metadata_small.sort_values("track_id").reset_index(drop=True)
genres_small_sorted = sorted(metadata_small["genre"].unique())
genre_to_idx_small = {g: i for i, g in enumerate(genres_small_sorted)}
metadata_small["label"] = metadata_small["genre"].map(genre_to_idx_small).astype(int)

print("\nSmall genre distribution:")
print(metadata_small["genre"].value_counts().sort_index())
print("\nSmall split distribution:")
print(metadata_small["split"].value_counts())


In [ ]:
print("=== Cleaning medium ===")
metadata_medium = metadata_all.copy()

# FMA convention: medium includes small. Use isin so the cleaned medium frame has all
# medium tracks plus the small tracks, matching the FMA paper's medium definition.
metadata_medium = keep(
    metadata_medium["subset"].isin(["small", "medium"]),
    metadata_medium,
    "subset in {small, medium}",
)
metadata_medium = keep(metadata_medium["genre"].notna(), metadata_medium, "genre not null")
metadata_medium = keep(~metadata_medium["track_id"].isin(FAILED), metadata_medium, "not in FAILED list")

is_small = metadata_medium["subset"] == "small"
if metadata_medium.loc[is_small, "audio_path"].map(bool).any():
    # If only fma_small/ is available, medium-only audio_path values are blank. That is fine
    # for tabular Random Forest training because features.csv covers those tracks.
    file_ok_or_medium_only = metadata_medium["audio_path"].map(is_file_nonempty) | (~is_small)
    metadata_medium = keep(file_ok_or_medium_only, metadata_medium, "small-portion audio present and non-empty")
else:
    print("No small audio directory found; skipping small-portion audio presence check.")

metadata_medium = metadata_medium.sort_values("track_id").reset_index(drop=True)
genres_medium_sorted = sorted(metadata_medium["genre"].unique())
genre_to_idx_medium = {g: i for i, g in enumerate(genres_medium_sorted)}
metadata_medium["label"] = metadata_medium["genre"].map(genre_to_idx_medium).astype(int)

print("\nMedium genre distribution:")
print(metadata_medium["genre"].value_counts().sort_index())
print("\nMedium split distribution:")
print(metadata_medium["split"].value_counts())


## 4. Save the cleaned CSVs

Both subset CSV families are written every run:
- `tracks_clean_{subset}.csv` — the full cleaned frame.
- `tracks_clean_{subset}_{split}.csv` — per-split slices so training notebooks can read just what they need.
- `genre_to_idx_{subset}.csv` — the genre to label mapping.


In [ ]:
def save_clean_csvs(frame: pd.DataFrame, subset_name: str, genre_to_idx: dict[str, int]) -> None:
    base_path = PROCESSED_DIR / f"tracks_clean_{subset_name}.csv"
    frame.to_csv(base_path, index=False)
    print(f"Wrote {base_path} ({len(frame):,} rows)")

    for split_name in ("training", "validation", "test"):
        split_df = frame[frame["split"] == split_name]
        split_path = PROCESSED_DIR / f"tracks_clean_{subset_name}_{split_name}.csv"
        split_df.to_csv(split_path, index=False)
        print(f"Wrote {split_path} ({len(split_df):,} rows)")

    genre_map_path = PROCESSED_DIR / f"genre_to_idx_{subset_name}.csv"
    pd.DataFrame(
        sorted(genre_to_idx.items(), key=lambda kv: kv[1]),
        columns=["genre", "label"],
    ).to_csv(genre_map_path, index=False)
    print(f"Wrote {genre_map_path}")


CLEANED_SUBSETS = {
    "small": (metadata_small, genre_to_idx_small),
    "medium": (metadata_medium, genre_to_idx_medium),
}

for subset_name, (frame, genre_map) in CLEANED_SUBSETS.items():
    save_clean_csvs(frame, subset_name, genre_map)
    print()

# Spectrograms are produced only for the selected subset.
metadata, genre_to_idx = CLEANED_SUBSETS[DATASET_SIZE]


## 5. Spectrogram settings

Spectrogram extraction uses the subset selected by `DATASET_SIZE`. If `DATASET_SIZE == "medium"` but `fma_medium/` is not available, the cleaned medium CSVs are still written and the spectrogram cells skip cleanly.

The saved spectrograms use the same numbers as `audio_encoder.ipynb` / `cnn_vs_transformer.ipynb` so they are drop-in compatible:

- 16 kHz mono
- 10 s clip (centered) -> 160,000 samples
- STFT: `n_fft=400`, `hop=160` (about 25 ms window, 10 ms hop)
- 64 mel bins, log power scale, per-clip mean/std normalization
- Output shape: `(1, 64, 1001)` saved as `float32`


In [ ]:
if EXTRACT_SPECTROGRAMS:
    SAMPLE_RATE = 16_000
    CLIP_SECONDS = 10
    NUM_SAMPLES = SAMPLE_RATE * CLIP_SECONDS
    N_FFT = 400
    HOP_LENGTH = 160
    N_MELS = 64

    mel_transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=SAMPLE_RATE,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
    ).to(device)
    to_db = torchaudio.transforms.AmplitudeToDB(stype="power").to(device)


    def load_clip(path: Path) -> torch.Tensor:
        """Read an MP3, convert to mono 16 kHz, center-crop or pad to NUM_SAMPLES."""
        audio, sample_rate = sf.read(path, dtype="float32", always_2d=True)
        waveform = torch.from_numpy(audio.T).mean(dim=0, keepdim=True)

        if sample_rate != SAMPLE_RATE:
            waveform = torchaudio.functional.resample(waveform, sample_rate, SAMPLE_RATE)

        if waveform.shape[1] > NUM_SAMPLES:
            start = (waveform.shape[1] - NUM_SAMPLES) // 2
            waveform = waveform[:, start:start + NUM_SAMPLES]
        elif waveform.shape[1] < NUM_SAMPLES:
            waveform = torch.nn.functional.pad(waveform, (0, NUM_SAMPLES - waveform.shape[1]))

        waveform = torch.nan_to_num(waveform, nan=0.0, posinf=0.0, neginf=0.0)
        return waveform.clamp(-1.0, 1.0)


    @torch.no_grad()
    def waveform_to_spec(waveform: torch.Tensor) -> torch.Tensor:
        waveform = waveform.to(device)
        spec = to_db(mel_transform(waveform))
        spec = torch.nan_to_num(spec, nan=0.0, posinf=0.0, neginf=0.0)
        mean = spec.mean(dim=(-2, -1), keepdim=True)
        std = spec.std(dim=(-2, -1), keepdim=True).clamp_min(1e-6)
        spec = (spec - mean) / std
        return torch.nan_to_num(spec, nan=0.0, posinf=0.0, neginf=0.0).cpu()


    # Sanity check on one clip.
    sample_row = metadata.iloc[0]
    sample_wave = load_clip(Path(sample_row["audio_path"]))
    sample_spec = waveform_to_spec(sample_wave)
    print(f"waveform shape: {tuple(sample_wave.shape)}")
    print(f"spectrogram shape: {tuple(sample_spec.shape)}")
    print(f"sample track: {sample_row['track_id']} ({sample_row['genre']})")
else:
    print(f"Skipping {DATASET_SIZE} spectrogram setup because no matching audio folder was found.")


## 6. Extract and save every spectrogram

When enabled, output layout mirrors the selected audio folder:

```
fma_preprocessed/
  spectrograms/
    000/000002.npy
    000/000005.npy
    ...
```

Already-extracted spectrograms are skipped so the cell is safe to re-run. Tracks that throw during decode/STFT are recorded in a final `dropped` list and removed from the manifest.


In [ ]:
def track_id_to_spec_path(track_id: int) -> Path:
    filename = f"{int(track_id):06d}.npy"
    return SPECTROGRAM_DIR / filename[:3] / filename


# Toggle to False to recompute everything from scratch.
SKIP_EXISTING = True

spec_paths = []
dropped_rows = []
n_skipped = 0
n_written = 0

if EXTRACT_SPECTROGRAMS:
    for row in tqdm(metadata.itertuples(index=False), total=len(metadata), desc=f"Extracting {DATASET_SIZE}"):
        out_path = track_id_to_spec_path(row.track_id)
        out_path.parent.mkdir(parents=True, exist_ok=True)

        if SKIP_EXISTING and out_path.exists():
            spec_paths.append(str(out_path))
            n_skipped += 1
            continue

        try:
            waveform = load_clip(Path(row.audio_path))
            spec = waveform_to_spec(waveform).numpy().astype(np.float32)
        except Exception as exc:
            dropped_rows.append({
                "track_id": row.track_id,
                "audio_path": str(row.audio_path),
                "error": f"{type(exc).__name__}: {exc}",
            })
            spec_paths.append(None)
            continue

        np.save(out_path, spec)
        spec_paths.append(str(out_path))
        n_written += 1

    print(f"\nWrote {n_written:,} new spectrograms, reused {n_skipped:,} existing, "
          f"dropped {len(dropped_rows):,} on error.")
    if dropped_rows:
        print(pd.DataFrame(dropped_rows).head(10).to_string(index=False))
else:
    print(f"Skipping {DATASET_SIZE} spectrogram extraction because no matching audio folder was found.")


## 7. Save the spectrogram manifest

When spectrogram extraction runs, `spectrograms_manifest.csv` points to the selected subset only: one row per usable clip with the `.npy` path, label, split, and original audio path.


In [ ]:
if EXTRACT_SPECTROGRAMS:
    manifest = metadata.copy()
    manifest["spectrogram_path"] = spec_paths
    manifest = manifest[manifest["spectrogram_path"].notna()].reset_index(drop=True)

    manifest_columns = [
        "track_id", "split", "genre", "label",
        "spectrogram_path", "audio_path",
        "duration", "bit_rate", "title", "artist",
    ]
    manifest = manifest[manifest_columns]

    manifest_path = PROCESSED_DIR / "spectrograms_manifest.csv"
    manifest.to_csv(manifest_path, index=False)
    print(f"Wrote {manifest_path} ({len(manifest):,} rows)")

    for split_name in ("training", "validation", "test"):
        split_manifest = manifest[manifest["split"] == split_name]
        split_path = PROCESSED_DIR / f"spectrograms_manifest_{split_name}.csv"
        split_manifest.to_csv(split_path, index=False)
        print(f"Wrote {split_path} ({len(split_manifest):,} rows)")

    if dropped_rows:
        dropped_path = PROCESSED_DIR / "spectrogram_extraction_errors.csv"
        pd.DataFrame(dropped_rows).to_csv(dropped_path, index=False)
        print(f"Wrote {dropped_path} ({len(dropped_rows):,} rows)")
else:
    manifest = pd.DataFrame()
    print(f"Skipping {DATASET_SIZE} spectrogram manifest because no spectrograms were extracted.")


## 8. Quick sanity check

When spectrogram extraction runs, load one selected-subset `.npy` back and confirm the shape/stats match the in-memory tensor we generated above.


In [ ]:
if EXTRACT_SPECTROGRAMS and not manifest.empty:
    first = manifest.iloc[0]
    loaded = np.load(first["spectrogram_path"])
    print(f"track_id: {first['track_id']} | genre: {first['genre']} | split: {first['split']}")
    print(f"shape: {loaded.shape} | dtype: {loaded.dtype}")
    print(f"mean: {loaded.mean():.4f} | std: {loaded.std():.4f} | "
          f"min: {loaded.min():.2f} | max: {loaded.max():.2f}")
else:
    print("No spectrogram sanity check to run for this dataset size.")
